In [1]:
import gc
import uproot
import numpy as np
import awkward as ak
from tqdm import tqdm
from coffea import processor
from coffea.nanoevents import PFNanoAODSchema

In [2]:
base_dir = "/hpcwork/rwth1244/PFNano/examples/"
files = [base_dir+i for i in ["QCD_HT100to200.root", "ttsemileptonic.root"]]

In [3]:
def pfnano_to_array(rootfile, isMC):
    print('Doing cleaning, isMC = ',isMC)
    
    feature_edges = []

    # Global
    feature_names = ['Jet_pt', 'Jet_eta',
                    'Jet_DeepJet_nCpfcand','Jet_DeepJet_nNpfcand',
                    'Jet_DeepJet_nsv','Jet_DeepJet_npv',
                    'Jet_DeepCSV_trackSumJetEtRatio',
                    'Jet_DeepCSV_trackSumJetDeltaR',
                    'Jet_DeepCSV_vertexCategory',
                    'Jet_DeepCSV_trackSip2dValAboveCharm',
                    'Jet_DeepCSV_trackSip2dSigAboveCharm',
                    'Jet_DeepCSV_trackSip3dValAboveCharm',
                    'Jet_DeepCSV_trackSip3dSigAboveCharm',
                    'Jet_DeepCSV_jetNSelectedTracks',
                    'Jet_DeepCSV_jetNTracksEtaRel'
                    ]
    feature_edges.append(len(feature_names))
    # CPF
    cpf = [[f'Jet_DeepJet_Cpfcan_BtagPf_trackEtaRel_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackPtRel_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackPPar_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackDeltaR_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackPParRatio_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackSip2dVal_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackSip2dSig_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackSip3dVal_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackSip3dSig_{i}',
            f'Jet_DeepJet_Cpfcan_BtagPf_trackJetDistVal_{i}',
            f'Jet_DeepJet_Cpfcan_ptrel_{i}',
            f'Jet_DeepJet_Cpfcan_drminsv_{i}',
            f'Jet_DeepJet_Cpfcan_VTX_ass_{i}',
            f'Jet_DeepJet_Cpfcan_puppiw_{i}',
            f'Jet_DeepJet_Cpfcan_chi2_{i}',
            f'Jet_DeepJet_Cpfcan_quality_{i}'] for i in range(25)]
    feature_names.extend([item for sublist in cpf for item in sublist])
    feature_edges.append(len(feature_names))
    # NPF
    npf = [[f'Jet_DeepJet_Npfcan_ptrel_{i}',
            f'Jet_DeepJet_Npfcan_deltaR_{i}',
            f'Jet_DeepJet_Npfcan_isGamma_{i}',
            f'Jet_DeepJet_Npfcan_HadFrac_{i}',
            f'Jet_DeepJet_Npfcan_drminsv_{i}',
            f'Jet_DeepJet_Npfcan_puppiw_{i}'] for i in range(25)]
    feature_names.extend([item for sublist in npf for item in sublist])
    feature_edges.append(len(feature_names))
    # VTX
    vtx = [[f'Jet_DeepJet_sv_pt_{i}',
            f'Jet_DeepJet_sv_deltaR_{i}',
            f'Jet_DeepJet_sv_mass_{i}',
            f'Jet_DeepJet_sv_ntracks_{i}',
            f'Jet_DeepJet_sv_chi2_{i}',
            f'Jet_DeepJet_sv_normchi2_{i}',
            f'Jet_DeepJet_sv_dxy_{i}',
            f'Jet_DeepJet_sv_dxysig_{i}',
            f'Jet_DeepJet_sv_d3d_{i}',
            f'Jet_DeepJet_sv_d3dsig_{i}',
            f'Jet_DeepJet_sv_costhetasvpv_{i}',
            f'Jet_DeepJet_sv_enratio_{i}'] for i in range(4)]
    feature_names.extend([item for sublist in vtx for item in sublist])
    feature_edges.append(len(feature_names))
    
    number_of_features = len(feature_names)
    
    if isMC == True and targets_necessary:
        # flavour definition for PFNano based on: https://indico.cern.ch/event/739204/#3-deepjet-overview
        if 'Jet_FlavSplit' in rootfile['Events'].keys():
            feature_names.extend(['Jet_FlavSplit'])
        else:
            feature_names.extend(['Jet_hadronFlavour','Jet_partonFlavour','Jet_nBHadrons'])
     
    print('Events:', rootfile['Events'].num_entries)
    
    # go through a specified number of events, and get the information (awkward-arrays) for the keys specified above
    for data in rootfile['Events'].iterate(feature_names, step_size=rootfile['Events'].num_entries, library='ak'):
        break
    
    # creating an array to store all the columns with their entries per jet, flatten per-event -> per-jet
    # this works ONLY because the number of jets per event will be accessible in the analyzer
    datacolumns = np.zeros((number_of_features+1, len(ak.flatten(data['Jet_pt'], axis=1))))
    #print(len(datacolumns))

    for featureindex in range(number_of_features):
        a = ak.flatten(data[feature_names[featureindex]], axis=1) # flatten along first inside to get jets
        datacolumns[featureindex] = ak.to_numpy(a)

    if isMC == True and targets_necessary:
        if 'Jet_FlavSplit' in rootfile['Events'].keys():
            flavsplit = ak.to_numpy(ak.flatten(data['Jet_FlavSplit'], axis=1))
            # if the list specified below was exhaustive, the -1 would get overwritten all the time
            #target_class = np.full_like(flavsplit, -1)                                                         # initialize
            # but it isn't the case, there are undefined jet flavors, therefore set to something that could be used later
            target_class = np.full_like(flavsplit, 1)                                                         # initialize
            target_class = np.where(flavsplit == 500, 0, target_class)                                                       # b
            target_class = np.where(np.bitwise_or(flavsplit == 510, flavsplit == 511), 1, target_class)                      # bb
            target_class = np.where(np.bitwise_or(flavsplit == 520, flavsplit == 521), 2, target_class)                      # leptonicb
            target_class = np.where(np.bitwise_or(flavsplit == 400, flavsplit == 410, flavsplit == 411), 3, target_class)    # c
            target_class = np.where(np.bitwise_or(flavsplit == 1, flavsplit == 2), 4, target_class)                          # uds
            target_class = np.where(flavsplit == 0, 5, target_class)                                                         # g
            del flavsplit
            gc.collect()
        else: # backup case for samples that don't have fine grained target definition available
            hadronFlav = ak.to_numpy(ak.flatten(data['Jet_hadronFlavour'], axis=1))
            partonFlav = ak.to_numpy(ak.flatten(data['Jet_partonFlavour'], axis=1))
            nBHadrons = ak.to_numpy(ak.flatten(data['Jet_nBHadrons'], axis=1))
            target_class = np.full_like(hadronFlav, 2)                                                         # initialize
            target_class = np.where(np.bitwise_and(hadronFlav == 5, nBHadrons == 1), 0, target_class)                        # b
            target_class = np.where(np.bitwise_and(hadronFlav == 5, nBHadrons > 1.5), 1, target_class)                       # bb
            #target_class = np.where(np.bitwise_or(flavsplit == 520, flavsplit == 521), 2, target_class)                     # leptonicb
            target_class = np.where(hadronFlav == 4, 3, target_class)                                                        # c
            target_class = np.where(np.bitwise_and(hadronFlav != 5, hadronFlav != 4), 4, target_class)                       # uds
            target_class = np.where(np.bitwise_and(hadronFlav != 5, hadronFlav != 4, partonFlav == 21), 5, target_class)     # g
            del hadronFlav
            del partonFlav
            del nBHadrons
            gc.collect()
        datacolumns[number_of_features] = target_class
        
    datavectors = datacolumns.transpose()
    print('Jets:', len(datavectors))    
    # shape of datavectors: number of jets, number of features  +    1
    #                                             inputs           target
    #                                       (both data and MC)   (MC only)
    # Maybe ToDo: wondering whether we need to clean the features like it was done for DeepCSV, the ShallowTagInfos are contained in DeepJet inputs!
    return datavectors, feature_edges

print("Dataset construction")
dataset = np.array([])
for fi in tqdm(files):
    print(fi)
    file = uproot.open(f"{fi}")
    targets_necessary = True
    output, feature_edges = pfnano_to_array(file, True)
    if len(dataset) == 0:
        dataset = output
    else:
        dataset = np.append(dataset, output, axis=0)
    print("shape:  ", output.shape)
    print("targets:", output[0:30,-1])
print("dataset shape:", dataset.shape)
np.save("/hpcwork/rwth1244/PFNano/examples/pfnano_to_array.npy", dataset)

Dataset construction


  0%|                                                     | 0/2 [00:00<?, ?it/s]

/hpcwork/rwth1244/PFNano/examples/QCD_HT100to200.root
Doing cleaning, isMC =  True
Events: 3998


 50%|██████████████████████▌                      | 1/2 [00:05<00:05,  5.01s/it]

Jets: 20957
shape:   (20957, 614)
targets: [5. 1. 1. 5. 1. 5. 1. 5. 1. 5. 5. 5. 5. 5. 5. 4. 5. 5. 4. 4. 5. 4. 1. 5.
 5. 5. 5. 1. 5. 4.]
/hpcwork/rwth1244/PFNano/examples/ttsemileptonic.root
Doing cleaning, isMC =  True
Events: 4019
Jets: 33079


100%|█████████████████████████████████████████████| 2/2 [00:10<00:00,  5.31s/it]

shape:   (33079, 614)
targets: [4. 5. 2. 3. 2. 1. 1. 1. 0. 2. 5. 4. 1. 1. 5. 4. 4. 1. 0. 4. 5. 4. 3. 0.
 5. 4. 1. 0. 4. 1.]
dataset shape: (54036, 614)


In [4]:
sample_dict = {"qcd": ["/hpcwork/rwth1244/PFNano/examples/QCD_HT100to200.root"], "tt": ["/hpcwork/rwth1244/PFNano/examples/ttsemileptonic.root"]}

In [5]:
feature_names = ["pt", "eta", "DeepJet_nCpfcand", "DeepJet_nNpfcand", "DeepJet_nsv", "DeepJet_npv", "DeepCSV_trackSumJetEtRatio", "DeepCSV_trackSumJetDeltaR", "DeepCSV_vertexCategory", "DeepCSV_trackSip2dValAboveCharm", "DeepCSV_trackSip2dSigAboveCharm", "DeepCSV_trackSip3dValAboveCharm", "DeepCSV_trackSip3dSigAboveCharm", "DeepCSV_jetNSelectedTracks", "DeepCSV_jetNTracksEtaRel"]
feature_edges.append(len(feature_names))

cpf = [[f"DeepJet_Cpfcan_BtagPf_trackEtaRel_{i}", f"DeepJet_Cpfcan_BtagPf_trackPtRel_{i}", f"DeepJet_Cpfcan_BtagPf_trackPPar_{i}", f"DeepJet_Cpfcan_BtagPf_trackDeltaR_{i}", f"DeepJet_Cpfcan_BtagPf_trackPParRatio_{i}", f"DeepJet_Cpfcan_BtagPf_trackSip2dVal_{i}", f"DeepJet_Cpfcan_BtagPf_trackSip2dSig_{i}", f"DeepJet_Cpfcan_BtagPf_trackSip3dVal_{i}", f"DeepJet_Cpfcan_BtagPf_trackSip3dSig_{i}", f"DeepJet_Cpfcan_BtagPf_trackJetDistVal_{i}", f"DeepJet_Cpfcan_ptrel_{i}", f"DeepJet_Cpfcan_drminsv_{i}", f"DeepJet_Cpfcan_VTX_ass_{i}", f"DeepJet_Cpfcan_puppiw_{i}", f"DeepJet_Cpfcan_chi2_{i}", f"DeepJet_Cpfcan_quality_{i}"] for i in range(25)]
feature_names.extend([item for sublist in cpf for item in sublist])
feature_edges.append(len(feature_names))

npf = [[f"DeepJet_Npfcan_ptrel_{i}", f"DeepJet_Npfcan_deltaR_{i}", f"DeepJet_Npfcan_isGamma_{i}", f"DeepJet_Npfcan_HadFrac_{i}", f"DeepJet_Npfcan_drminsv_{i}", f"DeepJet_Npfcan_puppiw_{i}"] for i in range(25)]
feature_names.extend([item for sublist in npf for item in sublist])
feature_edges.append(len(feature_names))

vtx = [[f"DeepJet_sv_pt_{i}", f"DeepJet_sv_deltaR_{i}", f"DeepJet_sv_mass_{i}", f"DeepJet_sv_ntracks_{i}", f"DeepJet_sv_chi2_{i}", f"DeepJet_sv_normchi2_{i}", f"DeepJet_sv_dxy_{i}", f"DeepJet_sv_dxysig_{i}", f"DeepJet_sv_d3d_{i}", f"DeepJet_sv_d3dsig_{i}", f"DeepJet_sv_costhetasvpv_{i}", f"DeepJet_sv_enratio_{i}"] for i in range(4)]
feature_names.extend([item for sublist in vtx for item in sublist])
feature_edges.append(len(feature_names))

feature_names.append("truth")
feature_edges.append(len(feature_names))

In [6]:
def empty_column_accumulator():
    return processor.column_accumulator(np.array([],dtype=np.float64))
def array_accumulator():
    return processor.defaultdict_accumulator(empty_column_accumulator)

class DeepJet_DataPreprocessing(processor.ProcessorABC):
    def __init__(self, deepjet_features):
        self.deepjet_features = deepjet_features
        self._accumulator = processor.dict_accumulator({})
    
    @property
    def accumulator(self):
        return self._accumulator

    def process(self, events):
        dataset = events.metadata["dataset"]
        output = self.accumulator
        for f in self.deepjet_features[:-1]:
            output[f"Jet_{f}"] = processor.column_accumulator(ak.to_numpy(ak.flatten(events["Jet"][f"{f}"], axis=1)))
        flavsplit = ak.to_numpy(ak.flatten(events["Jet"]["FlavSplit"], axis=1))
        target_class = np.full_like(flavsplit, 1)
        target_class = np.where(flavsplit == 500, 0, target_class)                                                       # b
        target_class = np.where(np.bitwise_or(flavsplit == 510, flavsplit == 511), 1, target_class)                      # bb
        target_class = np.where(np.bitwise_or(flavsplit == 520, flavsplit == 521), 2, target_class)                      # leptonicb
        target_class = np.where(np.bitwise_or(flavsplit == 400, flavsplit == 410, flavsplit == 411), 3, target_class)    # c
        target_class = np.where(np.bitwise_or(flavsplit == 1, flavsplit == 2), 4, target_class)                          # uds
        target_class = np.where(flavsplit == 0, 5, target_class)
        output[f"Jet_{self.deepjet_features[-1]}"] = processor.column_accumulator(target_class)
        return {dataset: output}

    def postprocess(self, accumulator):
        pass
        #return accumulator

In [7]:
futures_run = processor.Runner(executor = processor.FuturesExecutor(compression=None, workers=1), schema=PFNanoAODSchema, chunksize=1000) #, maxchunks=1)

out = futures_run(sample_dict, "Events", processor_instance=DeepJet_DataPreprocessing(feature_names))
#out

Output()

Output()

In [8]:
out

{'tt': {'Jet_pt': column_accumulator(array([60.9375  , 41.71875 , 36.21875 , ..., 19.328125, 19.296875,
         17.90625 ], dtype=float32)),
  'Jet_eta': column_accumulator(array([-0.5916748 ,  1.8769531 ,  0.40020752, ..., -2.855957  ,
         -1.6103516 ,  2.6391602 ], dtype=float32)),
  'Jet_DeepJet_nCpfcand': column_accumulator(array([5, 6, 2, ..., 3, 4, 5], dtype=int32)),
  'Jet_DeepJet_nNpfcand': column_accumulator(array([6, 5, 1, ..., 4, 3, 0], dtype=int32)),
  'Jet_DeepJet_nsv': column_accumulator(array([0, 1, 1, ..., 0, 0, 1], dtype=int32)),
  'Jet_DeepJet_npv': column_accumulator(array([23, 23, 23, ..., 39, 39, 39], dtype=int32)),
  'Jet_DeepCSV_trackSumJetEtRatio': column_accumulator(array([0.4873047 , 0.890625  , 0.4650879 , ..., 0.15795898, 0.        ,
         0.05047607], dtype=float32)),
  'Jet_DeepCSV_trackSumJetDeltaR': column_accumulator(array([0.06155396, 0.00583649, 0.03152466, ..., 0.2421875 , 3.1640625 ,
         0.18835449], dtype=float32)),
  'Jet_DeepCSV_ver

In [9]:
features = np.stack([np.concatenate([out[f"{datasets}"][f"Jet_{feature}"].value for datasets in sample_dict.keys()]) for feature in feature_names], axis=1)

In [10]:
np.shape(features)

(54036, 614)

In [11]:
np.save("/hpcwork/rwth1244/PFNano/examples/coffea.npy", features)

In [12]:
data_old = np.load("/hpcwork/rwth1244/PFNano/examples/pfnano_to_array.npy")
data_new = np.load("/hpcwork/rwth1244/PFNano/examples/coffea.npy")

In [13]:
np.sum(data_old-data_new)

0.0